# Krylov solvers compared

mtl5 binds ten Krylov solvers and eight preconditioners, and any solver takes
any preconditioner. Which to use depends on the matrix: symmetric or not,
definite or not. This notebook runs them on the C++ example's two systems, a
nonsymmetric one and a symmetric indefinite one, and on a harder indefinite
system where the choice matters. It then measures what the incomplete
factorizations buy on the Poisson matrices, the comparison that
`solve_three_ways.cpp` previews in its last part.

**Uses:** `mtl5.sparse.iterative_solve` and the named solvers
(`cgs`, `bicgstab`, `bicgstab_ell`, `gmres`, `bicg`, `qmr`, `tfqmr`,
`idr_s`, `minres`, `cg`), the preconditioners `diagonal`, `ic0`, `ilu0`,
`ssor`, `full_output=`, and `mtl5.generators`.

**From C++:** [`phase13_krylov_multigrid/krylov_comparison.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase13_krylov_multigrid/krylov_comparison.cpp), [`phase05_dense_decompositions/solve_three_ways.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/phase05_dense_decompositions/solve_three_ways.cpp) (Part 3).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp

import mtl5
import mtl5.sparse as ms

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

## A nonsymmetric system

The C++ example's convection–diffusion matrix has 2 on the diagonal, −1.1
below it and −0.9 above it, for n = 50 and b = 1. Every solver for
nonsymmetric systems converges on it. CG and MINRES are left out, because
they assume A = Aᵀ.

In [ ]:
n = 50
A_ns = ms.from_scipy(
    sp.diags([np.full(n - 1, -1.1), np.full(n, 2.0), np.full(n - 1, -0.9)], [-1, 0, 1]).tocsr()
)
b = np.ones(n)

runs = {
    "CGS": dict(solver="cgs"),
    "BiCGSTAB": dict(solver="bicgstab"),
    "BiCGSTAB(2)": dict(solver="bicgstab_ell", ell=2),
    "BiCGSTAB(4)": dict(solver="bicgstab_ell", ell=4),
    "BiCG": dict(solver="bicg"),
    "QMR": dict(solver="qmr"),
    "TFQMR": dict(solver="tfqmr"),
    "IDR(4)": dict(solver="idr_s", s=4),
    "GMRES(30)": dict(solver="gmres", restart=30),
}
reports = {}
for name, kw in runs.items():
    for pc in ("identity", "diagonal"):
        x, info, report = ms.iterative_solve(
            A_ns, b, M=getattr(ms, pc)(A_ns), rtol=1e-10, maxiter=1000, full_output=True, **kw
        )
        assert info == 0, (name, pc)
        reports[name, pc] = report

print(f"{'solver':>12}  iterations (identity / diagonal)   final relative residual")
for name in runs:
    a, d = reports[name, "identity"], reports[name, "diagonal"]
    counts = f"{a.iterations:5d} / {d.iterations:<5d}"
    print(f"{name:>12}  {counts}                    {a.relative_residual:.1e}")

Three things stand out.

- **An iteration isn't a unit of work.** Each method counts its own kind of
  step. BiCGSTAB(ℓ) counts one cycle of ℓ BiCG steps as an iteration, which is
  why BiCGSTAB(4) shows about half the count of BiCGSTAB(2). BiCGSTAB and CGS
  multiply by A twice per iteration, BiCG and QMR once by A and once by Aᵀ,
  and GMRES once. Compare iteration counts within a method, not across
  methods.
- **The diagonal preconditioner changes nothing**, except for GMRES. The
  diagonal of A is constant, so M = diag(A) is 2I, a scalar multiple of the
  identity that every other solver here is invariant to. GMRES is
  left-preconditioned and tests ‖M⁻¹r‖ against ‖b‖, so scaling by 1/2
  changes when it stops (stillwater-sc/mtl5#936).
- **Restarting costs GMRES.** GMRES(30) needs 271 iterations: each restart
  discards the Krylov space it has built.

The histories show how differently they get there. GMRES's residual never
increases, because it minimizes it. CGS squares the BiCG polynomial and with
it BiCG's bumps, here a climb of five orders of magnitude before it
converges. BiCGSTAB(ℓ) is the smoothest of the BiCG family.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.8))
shown = ["CGS", "BiCGSTAB", "BiCGSTAB(4)", "BiCG", "GMRES(30)"]
for color, name in zip(SERIES + ["#8a5cd6"], shown):
    hist = reports[name, "identity"].history
    ax.semilogy(hist / hist[0], color=color, label=name)
ax.set_xlabel("iteration")
ax.set_ylabel("tested residual / ||b||")
ax.set_xlim(0, 120)
ax.set_title("Nonsymmetric convection–diffusion, n = 50", fontsize=10)
ax.legend(frameon=False, fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

for name in runs:
    assert reports[name, "identity"].relative_residual <= 1e-9, name
    if name != "GMRES(30)":
        assert reports[name, "identity"].iterations == reports[name, "diagonal"].iterations, name
it = {name: reports[name, "identity"].iterations for name in runs}
assert it["BiCGSTAB(4)"] < it["BiCGSTAB(2)"] < it["BiCGSTAB"]
assert it["GMRES(30)"] > 3 * it["BiCG"]

## Symmetric but indefinite

The C++ example's second matrix alternates 4 and −3 on the diagonal, with 1
beside it: symmetric, with half its eigenvalues negative. MINRES is the
method for that case. It minimizes the residual over the Krylov space, as
GMRES does, but with CG's short recurrence, because A is symmetric.

This particular matrix is a gentle test, though. Its condition number is
1.5, so every method converges quickly, even CG, which has no guarantee for
an indefinite matrix.

In [ ]:
d = np.where(np.arange(n) % 2 == 0, 4.0, -3.0)
A_si = ms.from_scipy(sp.diags([np.ones(n - 1), d, np.ones(n - 1)], [-1, 0, 1]).tocsr())
eigs = np.linalg.eigvalsh(ms.to_scipy(A_si).toarray())
kappa = np.abs(eigs).max() / np.abs(eigs).min()
print(f"{(eigs < 0).sum()} negative eigenvalues of {n}, condition number {kappa:.2f}")
for s in ("minres", "cg", "gmres"):
    _, info, report = ms.iterative_solve(A_si, b, solver=s, rtol=1e-10, full_output=True)
    print(f"{s:>7}: info {info}, {report.iterations} iterations")
    assert info == 0

The C++ example also runs MINRES with the diagonal preconditioner, and that
combination is not valid. MINRES needs a symmetric positive definite
preconditioner, because it works in the inner product defined by M⁻¹. Here
diag(A) has the entries 4 and −3, so M is indefinite, and MINRES breaks down.

In [ ]:
_, info, report = ms.minres(A_si, b, M=ms.diagonal(A_si), rtol=1e-10, full_output=True)
print(
    f"MINRES with M = diag(A): info {info} after {report.iterations} iterations, "
    f"relative residual {report.relative_residual:.1e}"
)
assert info != 0 and report.relative_residual > 1e-3

## An indefinite system where the method matters

A harder indefinite matrix comes from the Helmholtz equation, −u″ − σu = f.
On 100 points, with σ = 1000, the shift moves the 10 smallest eigenvalues of
the 1D Laplacian below zero.

- **MINRES** converges, and its residual never increases, by construction.
- **CG** converges too, here, but its residual jumps up and down along the
  way. On another right-hand side or shift it can break down when pᵀAp = 0.
- **GMRES(30)** loses its progress at every restart.

In [ ]:
m = 100
hm = 1.0 / (m + 1)
L = ms.to_scipy(mtl5.generators.laplacian_1d(m)) / hm**2
sigma = 1000.0
H = ms.from_scipy((L - sigma * sp.eye(m)).tocsr())
bh = np.ones(m)
print(f"negative eigenvalues: {(np.linalg.eigvalsh((L - sigma * sp.eye(m)).toarray()) < 0).sum()}")

helm = {}
for s in ("minres", "cg", "bicgstab", "gmres"):
    _, info, report = ms.iterative_solve(
        H, bh, solver=s, rtol=1e-10, maxiter=2000, full_output=True
    )
    helm[s] = (info, report)
    rel = report.relative_residual
    print(f"{s:>9}: info {info}, {report.iterations} iterations, relative residual {rel:.1e}")

fig, ax = plt.subplots(figsize=(6.4, 3.6))
for color, s in zip(SERIES, ("minres", "cg", "bicgstab", "gmres")):
    hist = helm[s][1].history
    on_top = {"zorder": 3, "linewidth": 2.2} if s == "minres" else {"linewidth": 1.0}
    label = {"minres": "MINRES", "cg": "CG", "bicgstab": "BiCGSTAB", "gmres": "GMRES(30)"}[s]
    ax.semilogy(hist / hist[0], color=color, label=label, **on_top)
ax.set_xlim(0, 200)
ax.set_xlabel("iteration")
ax.set_ylabel("tested residual / ||b||")
ax.set_title("Shifted Laplacian, 10 negative eigenvalues", fontsize=10)
ax.legend(frameon=False, fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

info, minres_report = helm["minres"]
assert info == 0 and minres_report.relative_residual <= 1e-9
hist = minres_report.history
assert (np.diff(hist) <= 1e-12 * hist[0]).all()  # MINRES: monotone
cg_hist = helm["cg"][1].history
assert (np.diff(cg_hist) > 0).any()  # CG: not monotone on an indefinite matrix
assert helm["gmres"][1].iterations > 5 * minres_report.iterations

## What a preconditioner buys

The last part of `solve_three_ways.cpp` compares ILU(0) with BiCGSTAB, IC(0)
with CG and the diagonal preconditioner with CG, on a 1D Laplacian of 20
points. On that matrix the comparison says little.

- **The incomplete factorizations are complete.** IC(0) and ILU(0) keep only
  the nonzeros already in A. A tridiagonal matrix's Cholesky and LU factors
  have no others, so nothing is dropped, and the solver converges in one
  iteration.
- **The diagonal is constant again,** so the Jacobi preconditioner is a scalar
  and changes nothing.

In [ ]:
n1 = 20
A1 = ms.from_scipy(ms.to_scipy(mtl5.generators.laplacian_1d(n1)) * (n1 + 1) ** 2)
b1 = np.ones(n1)
one_d = {}
for label, solver, pc in [
    ("CG", "cg", "identity"),
    ("CG + diagonal", "cg", "diagonal"),
    ("CG + IC(0)", "cg", "ic0"),
    ("BiCGSTAB", "bicgstab", "identity"),
    ("BiCGSTAB + ILU(0)", "bicgstab", "ilu0"),
]:
    _, info, report = ms.iterative_solve(
        A1, b1, solver=solver, M=getattr(ms, pc)(A1), rtol=1e-10, full_output=True
    )
    assert info == 0
    one_d[label] = report.iterations
print(one_d)
assert one_d["CG + IC(0)"] == 1 and one_d["BiCGSTAB + ILU(0)"] == 1
assert one_d["CG + diagonal"] == one_d["CG"]

The 2D Poisson matrix is where the factorizations become incomplete. Its
Cholesky factor fills in the band between the outer diagonals, and IC(0)
drops all of it. They still roughly halve the iterations, at every grid size.
But the count still doubles when the grid is refined, because no incomplete
factorization changes how the conditioning grows, as O(1/h²). Removing that
growth takes [multigrid](multigrid.ipynb).

In [ ]:
grid_sizes = [20, 40, 80]
table = {}
for g in grid_sizes:
    Ag = mtl5.generators.poisson2d(g, g)
    bg = np.ones(g * g)
    for label, solver, pc in [
        ("CG", "cg", "identity"),
        ("CG + SSOR", "cg", "ssor"),
        ("CG + IC(0)", "cg", "ic0"),
        ("BiCGSTAB", "bicgstab", "identity"),
        ("BiCGSTAB + ILU(0)", "bicgstab", "ilu0"),
    ]:
        _, info, report = ms.iterative_solve(
            Ag, bg, solver=solver, M=getattr(ms, pc)(Ag), rtol=1e-10, maxiter=5000, full_output=True
        )
        assert info == 0, (g, label)
        table[label, g] = report.iterations

labels = ["CG", "CG + SSOR", "CG + IC(0)", "BiCGSTAB", "BiCGSTAB + ILU(0)"]
fig, ax = plt.subplots(figsize=(6.0, 3.6))
for color, label in zip(SERIES + ["#8a5cd6"], labels):
    ax.plot(grid_sizes, [table[label, g] for g in grid_sizes], "o-", color=color, label=label)
ax.set_xlabel("grid points per side")
ax.set_ylabel("iterations to rtol 1e-10")
ax.set_xticks(grid_sizes)
ax.set_title("2D Poisson: preconditioners halve the count, not its growth", fontsize=10)
ax.legend(frameon=False, fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.show()

for label in labels:
    print(f"{label:>18}: " + "  ".join(f"{g}x{g}: {table[label, g]:4d}" for g in grid_sizes))
for g in grid_sizes:
    assert table["CG + IC(0)", g] < 0.6 * table["CG", g]
    assert table["BiCGSTAB + ILU(0)", g] < 0.6 * table["BiCGSTAB", g]
for label in labels:
    assert table[label, 80] > 1.6 * table[label, 40], label  # still grows with the grid

## From C++

| MTL5 (C++) | mtl5 (Python) |
| --- | --- |
| `itl::cgs`, `itl::bicgstab`, `itl::bicgstab_ell(..., ell)`, `itl::minres` | `ms.cgs`, `ms.bicgstab`, `ms.bicgstab_ell(..., ell=)`, `ms.minres`, or `ms.iterative_solve(..., solver=)` |
| `itl::pc::identity<M>`, `itl::pc::diagonal<M>` | `ms.identity(A)`, `ms.diagonal(A)` |
| `itl::pc::ilu_0<double>`, `itl::pc::ic_0<double>` | `ms.ilu0(A)`, `ms.ic0(A)` |
| `iter.iterations()`, the true residual computed by hand | `report.iterations`, `report.relative_residual` from `full_output=True` |
| `inserter<compressed2D<double>>`, entry by entry | `scipy.sparse.diags` and `ms.from_scipy`, or `mtl5.generators` |

The C++ program's MINRES with `pc::diagonal` on the alternating matrix is the
invalid pairing shown above: the preconditioner must be positive definite.